# Train VAE-GAN

Uses whole-demonstration partitions, a Neo VAE and three optimizers. Re-running resumes the last committed epoch. Increase `epochs` to extend training; use a new exp/run for configuration changes. Intermediate checkpoint retention is controlled by `keep_checkpoints`.


In [ ]:
from pathlib import Path
import sys
import torch
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

# Run from this notebook's sensorprocessing directory.
if str(Path("..").resolve()) not in sys.path:
    sys.path.insert(0, str(Path("..").resolve()))
from exp_run_config import Config
Config.PROJECTNAME = "BerryPicker"
from sensorprocessing.conv_vae_neo import ConvVAENeo, make_dataloaders
from sensorprocessing.vae_gan_training import train
from sensorprocessing.vae_gan_visualization import plot_history, plot_reconstructions, plot_images
from sensorprocessing.sp_factory import create_sp
from training_harness.checkpoints import model_file

experiment = "sensorprocessing_vae_gan"
run = "sp_vae_gan_128_256px"
expruns_path = None  # Optional existing external configuration directory
results_path = None  # Optional existing results directory
if expruns_path is not None:
    if not Path(expruns_path).is_dir():
        raise FileNotFoundError(expruns_path)
    Config().set_exprun_path(expruns_path)
    for group in [experiment, "sensorprocessing_conv_vae_neo", "demonstration", "robot_al5d"]:
        Config().copy_experiment(group)
if results_path is not None:
    if not Path(results_path).is_dir():
        raise FileNotFoundError(results_path)
    Config().set_results_path(results_path)
exp = Config().get_experiment(experiment, run, creation_style="exist-ok")
device = Config().runtime["device"]


In [ ]:
loaders = make_dataloaders(exp)
validation_dataset = loaders[1].dataset
images = torch.stack([validation_dataset[i] for i in range(min(4, len(validation_dataset)))]).to(device)
# Local CPU generator: these samples remain fixed without altering training RNG.
latent = torch.randn(len(images), exp["latent_size"], generator=torch.Generator().manual_seed(123)).to(device)


In [ ]:
def show_progress(model, history):
    clear_output(wait=True)
    print(f'Epoch {history[-1]["epoch"]}: {history[-1]["phase"]}')
    display(plot_history(history))
    display(plot_reconstructions(model, images, latent))
    plt.close("all")

last_model, history = train(exp, loaders=loaders, callback=show_progress)


## Inspect the best VAE

The raw export is Neo-compatible. Full training checkpoints contain the VAE, discriminator, all optimizer states and recovery metadata; they are not runtime exports.


In [ ]:
# Load the best exported VAE, not the last training state.
model = ConvVAENeo(exp).to(device)
model.load_state_dict(torch.load(model_file(exp), map_location=device, weights_only=True))
model.eval()
print(f"Loaded best VAE: {model_file(exp)}")

# Alternatively load the VAE portion of a retained intermediate full checkpoint:
# checkpoint_path = Path(exp["data_dir"]) / "checkpoints" / "epoch_000010.pt"
# checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
# model.load_state_dict({key.removeprefix("vae."): value
#                        for key, value in checkpoint["model_state_dict"].items()
#                        if key.startswith("vae.")})
# model.eval()


In [ ]:
display(plot_reconstructions(model, images, latent))
